In [1]:
import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [30]:
%load_ext autoreload
%autoreload 2

import random
import numpy as np
import torch
import Python.simfun as sim

seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
sigma2=1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------------
# 1. Simulate sparse BNN oracle data
# -----------------------------

X, y, feature_true, truth, sim_info = sim.simfun_bnn(
    n=160,
    p=100,
    seed=123,
    family="gaussian",
    sigma2=1.0,
    d_model=8,
    n_blocks=2,
    ffn_dims=[8, 8],
    n_active=4,
    bounded=None,
    center_y=True,
    attention_type="self",
    ffn_activation="relu",
    device=device,
)

print(sim.print_bnn_siminfo(sim_info))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Sparse attention-FFN BNN simulation summary:
  sim                   : attention_ffn_sparse_oracle
  seed                  : 123
  family                : gaussian
  n                     : 160
  p                     : 100
  n_active              : 4
  sigma2                : 1.0000
  sigma                 : 1.0000
  signal_var            : 8.4284
  outcome_var           : 8.8447
  input_dim             : 100
  d_model               : 8
  n_blocks              : 2
  ffn_dims              : [8, 8]
  out_dim               : 1
  attention_type        : self
  ffn_activation        : relu
  center_y              : True
  n_param               : 1105
  n_nonzero             : 105
  truth_density         : 0.0950

Active input features, zero-based:
  [1, 5, 58, 66]

Active state coordinates:
  active_state : [0, 1, 2, 7]

Active FFN units by residual block:
  block 0  : [1, 2, 5, 6]
  block 1  : [1, 2, 6

In [25]:
%load_ext autoreload
%autoreload 2

import Python.config as cfg
import Python.model2 as md
import Python.utils as ut
import Python.bnn_mcmc as mcmc

split_cfg = cfg.SplitConfig(
    train_frac=0.60,
    val_frac=0.20,
    test_frac=0.20,
    seed=seed,
)

indices = ut.make_split(X.shape[0], split_cfg)
splits = ut.split_data(X, y, indices, mode="tensor")

X_train = splits["X_train"]
y_train = splits["y_train"]

X_val = splits["X_val"]
y_val = splits["y_val"]

X_test = splits["X_test"]
y_test = splits["y_test"]


decoder = md.DSSAttentionFFNDecoder(
    input_dim=X_train.shape[1],
    d_model=8,
    n_blocks=2,
    ffn_dims=[8, 8],
    out_dim=1,
    lambda_w=1.0,
    lambda_b=1.0,
    bounded=None,
    attention_type="self",
    ffn_activation="relu",
).to(device)


mcmc_ref = mcmc.run_bnn_mcmc(
    decoder=decoder,
    X=X_train,
    y=y_train,
    family="gaussian",
    sigma2=1.0,
    truth=truth,
    N=10000,
    S_max=100,
    burnin=2000,
    thin=1,
    beta_eps=0.05,
    seed=123,
    print_every=500,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
mcmc_iter=00001 loglik=-124.740
mcmc_iter=00500 loglik=-130.073
mcmc_iter=01000 loglik=-138.589
mcmc_iter=01500 loglik=-152.535
mcmc_iter=02000 loglik=-127.497
mcmc_iter=02500 loglik=-138.061
mcmc_iter=03000 loglik=-139.211
mcmc_iter=03500 loglik=-142.573
mcmc_iter=04000 loglik=-149.075
mcmc_iter=04500 loglik=-140.875
mcmc_iter=05000 loglik=-139.549
mcmc_iter=05500 loglik=-145.989
mcmc_iter=06000 loglik=-132.501
mcmc_iter=06500 loglik=-149.001
mcmc_iter=07000 loglik=-139.799
mcmc_iter=07500 loglik=-132.356
mcmc_iter=08000 loglik=-132.154
mcmc_iter=08500 loglik=-136.182
mcmc_iter=09000 loglik=-144.821
mcmc_iter=09500 loglik=-132.798
mcmc_iter=10000 loglik=-142.657


In [31]:

import copy
import numpy as np
import pandas as pd
import torch

model = md.LaSTBNNVI(
    X=X_train,
    y=y_train,
    input_dim=X_train.shape[1],
    d_model=8,
    n_blocks=2,
    ffn_dims=8,
    out_dim=1,
    family="gaussian",
    sigma2=1.0,
    K_flow=8,
    flow_hidden_units=64,
    flow_hidden_layers=2,
    scale_clip=1.5,
    lambda_w=1.0,
    lambda_b=1.0,
    bounded=None,
    attention_type="self",
    ffn_activation="relu",
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

R_train = 16
R_eval = 200

beta_eps = 0.05
pip_cut = 0.5

eval_every = 250
min_epoch = 500
target_input_support = 4

history = []
states = []

prev_input_pip = None

for epoch in range(1, 6001):
    optimizer.zero_grad(set_to_none=True)

    loss = model.neg_elbo(R=R_train, elbo_beta=1.0)

    loss.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        5.0,
    ).item()

    optimizer.step()

    if epoch == 1 or epoch % eval_every == 0:
        with torch.no_grad():
            xi_eval, _ = model.sample_posterior(R_eval)

            pred_train = model.decoder(X_train, xi_eval).mean(dim=0)
            pred_val = model.decoder(X_val, xi_eval).mean(dim=0)
            pred_test = model.decoder(X_test, xi_eval).mean(dim=0)

            train_mse = ((pred_train - y_train) ** 2).mean().item()
            val_mse = ((pred_val - y_val) ** 2).mean().item()
            test_mse = ((pred_test - y_test) ** 2).mean().item()

            train_rmse = float(np.sqrt(train_mse))
            val_rmse = float(np.sqrt(val_mse))
            test_rmse = float(np.sqrt(test_mse))

            params = model.decoder.unpack(xi_eval, return_summary=False)

            E = params["E"]  # R x d_model x p

            input_active = (E.abs() > beta_eps).any(dim=1).float()
            input_pip = input_active.mean(dim=0)

            expected_input_support = input_pip.sum().item()
            hard_input_support = (input_pip > pip_cut).float().sum().item()

            p_clip = input_pip.clamp(1e-6, 1.0 - 1e-6)
            input_entropy = (
                -p_clip * torch.log(p_clip)
                - (1.0 - p_clip) * torch.log(1.0 - p_clip)
            ).mean().item()

            if prev_input_pip is None:
                input_pip_churn = np.nan
                input_selected_churn = np.nan
            else:
                input_pip_np = input_pip.detach().cpu().numpy()
                input_pip_churn = float(np.mean(np.abs(input_pip_np - prev_input_pip)))

                old_sel = prev_input_pip > pip_cut
                new_sel = input_pip_np > pip_cut
                input_selected_churn = float(np.mean(old_sel != new_sel))

            prev_input_pip = input_pip.detach().cpu().numpy()

            middle_selected = 0.0
            middle_total = 0.0

            for k in range(model.decoder.n_blocks):
                for key in [f"W1_{k}", f"b1_{k}", f"W2_{k}", f"b2_{k}"]:
                    epip = (params[key].abs() > beta_eps).float().mean(dim=0)
                    middle_selected += (epip > pip_cut).float().sum().item()
                    middle_total += epip.numel()

            middle_density = middle_selected / max(middle_total, 1.0)

            Wout_epip = (params["Wout"].abs() > beta_eps).float().mean(dim=0)
            bout_epip = (params["bout"].abs() > beta_eps).float().mean(dim=0)

            output_selected = (
                (Wout_epip > pip_cut).float().sum().item()
                + (bout_epip > pip_cut).float().sum().item()
            )

        state_id = len(states)

        history.append({
            "state_id": state_id,
            "epoch": int(epoch),
            "loss": float(loss.item()),
            "log_loss": float(np.log(max(loss.item(), 1e-12))),
            "grad_norm": float(grad_norm),
            "log_grad": float(np.log(max(grad_norm, 1e-12))),
            "train_mse": float(train_mse),
            "val_mse": float(val_mse),
            "test_mse": float(test_mse),
            "train_rmse": float(train_rmse),
            "val_rmse": float(val_rmse),
            "test_rmse": float(test_rmse),
            "expected_input_support": float(expected_input_support),
            "hard_input_support": float(hard_input_support),
            "input_entropy": float(input_entropy),
            "input_pip_churn": float(input_pip_churn),
            "input_selected_churn": float(input_selected_churn),
            "middle_selected": float(middle_selected),
            "middle_density": float(middle_density),
            "output_selected": float(output_selected),
        })

        states.append({
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        })

        print(
            f"epoch={epoch:04d} "
            f"loss={loss.item():.3f} "
            f"grad={grad_norm:.2f} "
            f"train_mse={train_mse:.3f} "
            f"val_mse={val_mse:.3f} "
            f"test_mse={test_mse:.3f} "
            f"Esoft={expected_input_support:.2f} "
            f"Ehard={hard_input_support:.0f} "
            f"churn={input_pip_churn:.3f} "
            f"mid_density={middle_density:.3f} "
            f"outS={output_selected:.0f}"
        )

epoch=0001 loss=4938.182 grad=150.59 train_mse=8.544 val_mse=11.467 test_mse=7.123 Esoft=0.79 Ehard=0 churn=nan mid_density=0.000 outS=0
epoch=0250 loss=552.898 grad=983.01 train_mse=8.550 val_mse=11.475 test_mse=7.125 Esoft=79.80 Ehard=100 churn=0.790 mid_density=0.059 outS=0
epoch=0500 loss=560.935 grad=804.47 train_mse=8.554 val_mse=11.440 test_mse=7.128 Esoft=50.00 Ehard=46 churn=0.298 mid_density=0.066 outS=0
epoch=0750 loss=539.342 grad=3188.03 train_mse=8.556 val_mse=11.455 test_mse=7.121 Esoft=75.71 Ehard=100 churn=0.257 mid_density=0.021 outS=0
epoch=1000 loss=532.499 grad=581.18 train_mse=8.543 val_mse=11.472 test_mse=7.121 Esoft=82.51 Ehard=100 churn=0.069 mid_density=0.146 outS=0
epoch=1250 loss=540.662 grad=584.91 train_mse=8.547 val_mse=11.463 test_mse=7.125 Esoft=79.67 Ehard=100 churn=0.037 mid_density=0.010 outS=0
epoch=1500 loss=526.673 grad=502.17 train_mse=8.607 val_mse=11.427 test_mse=7.141 Esoft=77.67 Ehard=100 churn=0.035 mid_density=0.024 outS=0
epoch=1750 loss=7

In [32]:
hist_df = pd.DataFrame(history).copy()

hist_df["loss_slope"] = hist_df["log_loss"].diff(2).abs() / 2.0
hist_df["grad_slope"] = hist_df["log_grad"].diff(2).abs() / 2.0

hist_df["loss_slope"] = hist_df["loss_slope"].fillna(hist_df["loss_slope"].median())
hist_df["grad_slope"] = hist_df["grad_slope"].fillna(hist_df["grad_slope"].median())
hist_df["input_pip_churn"] = hist_df["input_pip_churn"].fillna(hist_df["input_pip_churn"].median())

best_val = hist_df.loc[hist_df["epoch"] >= min_epoch, "val_mse"].min()

cand = hist_df[
    (hist_df["epoch"] >= min_epoch)
    & (hist_df["val_mse"] <= best_val * 1.03)
].copy()

if len(cand) == 0:
    cand = hist_df[hist_df["epoch"] >= min_epoch].copy()

cand["support_penalty"] = (
    cand["expected_input_support"] - float(target_input_support)
).abs() / float(target_input_support)

cand["collapse_penalty"] = 0.0
cand.loc[cand["expected_input_support"] < 0.5, "collapse_penalty"] += 2.0
cand.loc[cand["output_selected"] < 0.5, "collapse_penalty"] += 1.0

cand["score"] = (
    cand["val_mse"] / (best_val + 1e-12)
    + 0.15 * cand["loss_slope"] / (cand["loss_slope"].median() + 1e-12)
    + 0.15 * cand["grad_slope"] / (cand["grad_slope"].median() + 1e-12)
    + 0.25 * cand["input_pip_churn"] / (cand["input_pip_churn"].median() + 1e-12)
    + 0.25 * cand["support_penalty"]
    + 0.10 * cand["input_entropy"] / (cand["input_entropy"].median() + 1e-12)
    + cand["collapse_penalty"]
)

cand = cand.sort_values(["score", "epoch"]).reset_index(drop=True)

display(cand[[
    "state_id",
    "epoch",
    "score",
    "loss",
    "grad_norm",
    "val_mse",
    "test_mse",
    "expected_input_support",
    "hard_input_support",
    "input_pip_churn",
    "middle_density",
    "output_selected",
    "loss_slope",
    "grad_slope",
    "support_penalty",
    "collapse_penalty",
]].head(10))

chosen = cand.iloc[0]
chosen_state_id = int(chosen["state_id"])

model.load_state_dict({
    k: v.to(device)
    for k, v in states[chosen_state_id].items()
})

print("\nchosen checkpoint:")
display(chosen.to_frame().T)

summary = model.posterior_summary(R=1000, beta_eps=beta_eps)

print("decoder dim:", model.decoder.dim)
print("s_dim:", model.decoder.s_dim)
print("u_dim:", model.decoder.u_dim)
print("t_dim:", model.decoder.t_dim)
print("t_mean:", summary["t_mean"])

,state_id,epoch,score,loss,grad_norm,val_mse,test_mse,expected_input_support,hard_input_support,input_pip_churn,middle_density,output_selected,loss_slope,grad_slope,support_penalty,collapse_penalty
0,20,5000,6.444,516.930,321.378,11.463,7.123,63.355,100.0,0.204,0.007,0.0,1.907e-03,0.002,14.839,1.0
1,13,3250,6.593,518.042,545.637,11.474,7.117,67.450,100.0,0.058,0.021,0.0,3.125e-03,0.155,15.862,1.0
2,24,6000,6.646,512.991,288.738,11.459,7.126,68.035,100.0,0.148,0.038,0.0,2.112e-03,0.036,16.009,1.0
3,22,5500,6.726,515.163,310.019,11.243,7.249,71.355,100.0,0.122,0.010,0.0,1.712e-03,0.018,16.839,1.0
4,6,1500,7.150,526.673,502.169,11.427,7.141,77.670,100.0,0.035,0.024,0.0,5.500e-03,0.073,18.417,1.0
5,11,2750,7.153,521.289,400.210,11.466,7.124,78.575,100.0,0.088,0.094,0.0,2.847e-03,0.022,18.644,1.0
6,14,3500,7.263,524.607,535.362,11.467,7.123,74.580,100.0,0.072,0.003,0.0,7.263e-03,0.168,17.645,1.0
7,15,3750,7.411,518.600,356.129,11.407,7.155,81.290,100.0,0.069,0.024,0.0,5.391e-04,0.213,19.323,1.0
8,23,5750,7.439,512.657,299.062,11.484,7.130,82.790,100.0,0.114,0.010,0.0,2.336e-03,0.015,19.697,1.0
9,8,2000,7.466,522.902,447.001,11.459,7.126,84.590,100.0,0.037,0.014,0.0,3.593e-03,0.058,20.148,1.0



chosen checkpoint:


,state_id,epoch,loss,log_loss,grad_norm,log_grad,train_mse,val_mse,test_mse,train_rmse,val_rmse,test_rmse,expected_input_support,hard_input_support,input_entropy,input_pip_churn,input_selected_churn,middle_selected,middle_density,output_selected,loss_slope,grad_slope,support_penalty,collapse_penalty,score
0,20.0,5000.0,516.93,6.248,321.378,5.773,8.545,11.463,7.123,2.923,3.386,2.669,63.355,100.0,0.655,0.204,0.0,2.0,0.007,0.0,0.002,0.002,14.839,1.0,6.444


decoder dim: 2222
s_dim: 1105
u_dim: 1105
t_dim: 12
t_mean: tensor([ 0.7807, -0.0968,  0.2155, -0.0361,  0.2200,  0.0844,  0.1697,  0.2141,
         0.1375, -0.1099,  1.8957,  0.6658])


In [29]:
# -----------------------------
# Compact attention-FFN BNN posterior + selection report
# -----------------------------

import numpy as np
import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 80)

beta_eps = 0.05
pip_cut = 0.5
R_vi = 1000


# -----------------------------
# 1. Small utilities
# -----------------------------

def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def param_label_prefix(key):
    if key == "E":
        return "z", "x"
    if key == "e":
        return "row", "z"
    if key.startswith("W1_"):
        k = key.split("_")[1]
        return f"ff{k}_", "z"
    if key.startswith("b1_"):
        k = key.split("_")[1]
        return "row", f"ff{k}_"
    if key.startswith("W2_"):
        k = key.split("_")[1]
        return "z", f"ff{k}_"
    if key.startswith("b2_"):
        return "row", "z"
    if key == "Wout":
        return "y", "z"
    if key == "bout":
        return "row", "y"
    return "r", "c"


def make_df(x, row_prefix="r", col_prefix="c"):
    x = np.asarray(x)

    if x.ndim == 1:
        x = x[None, :]

    rows = [f"{row_prefix}{i}" for i in range(x.shape[0])]
    cols = [f"{col_prefix}{j}" for j in range(x.shape[1])]

    return pd.DataFrame(x, index=rows, columns=cols)


def hist_skl(x, y, bins=60, eps=1e-8):
    lo = min(np.quantile(x, 0.01), np.quantile(y, 0.01))
    hi = max(np.quantile(x, 0.99), np.quantile(y, 0.99))

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.nan

    px, edges = np.histogram(x, bins=bins, range=(lo, hi))
    py, _ = np.histogram(y, bins=edges)

    px = px.astype(float) + eps
    py = py.astype(float) + eps

    px = px / px.sum()
    py = py / py.sum()

    return 0.5 * (
        np.sum(px * np.log(px / py))
        + np.sum(py * np.log(py / px))
    )


def bern_js(p, q, eps=1e-8):
    p = np.clip(p, eps, 1.0 - eps)
    q = np.clip(q, eps, 1.0 - eps)
    m = 0.5 * (p + q)

    kl_pm = p * np.log(p / m) + (1 - p) * np.log((1 - p) / (1 - m))
    kl_qm = q * np.log(q / m) + (1 - q) * np.log((1 - q) / (1 - m))

    return 0.5 * (kl_pm + kl_qm)


def param_group(key):
    if key in {"E", "e"}:
        return "input"
    if key in {"Wout", "bout"}:
        return "output"
    if "_" in key:
        return "block_" + key.split("_")[1]
    return "other"


# -----------------------------
# 2. Flatten order and summaries
# -----------------------------

def get_param_meta(decoder):
    rows = []

    for item in decoder.param_specs:
        rows.append({
            "group": param_group(item["name"]),
            "block": item["block"],
            "param": item["name"],
            "shape": item["shape"],
            "start": int(item["start"]),
            "end": int(item["end"]),
            "n_param": int(np.prod(item["shape"])),
        })

    return rows


@torch.no_grad()
def get_vi_draws_and_summary(model, R=1000, beta_eps=0.05):
    xi, _ = model.sample_posterior(R)
    params = model.decoder.unpack(xi, return_summary=False)

    theta_parts = []
    theta_names = []
    block_sum = {}

    for item in model.decoder.param_specs:
        key = item["name"]

        val = params[key].reshape(R, -1)

        theta_parts.append(val.detach().cpu())

        for local_id in range(val.shape[1]):
            theta_names.append(f"{key}_{local_id}")

        block = val.reshape(R, *item["shape"]).detach().cpu().numpy()

        block_sum[f"{key}_mean"] = block.mean(axis=0)
        block_sum[f"{key}_epip"] = (np.abs(block) > beta_eps).mean(axis=0)

    theta_draws = torch.cat(theta_parts, dim=1).numpy()

    return theta_draws, theta_names, block_sum


def get_mcmc_block_summary(model, mcmc_obj, beta_eps=0.05):
    theta = mcmc_obj["theta_draws"]
    block_sum = {}

    pos = 0

    for item in model.decoder.param_specs:
        key = item["name"]
        n_elem = int(np.prod(item["shape"]))

        block = theta[:, pos:pos + n_elem].reshape(
            theta.shape[0],
            *item["shape"],
        )

        block_sum[f"{key}_mean"] = block.mean(axis=0)
        block_sum[f"{key}_epip"] = (np.abs(block) > beta_eps).mean(axis=0)

        pos += n_elem

    return block_sum


def get_truth_flat(model, truth):
    vals = []
    names = []

    for item in model.decoder.param_specs:
        key = item["name"]

        if key not in truth:
            raise KeyError(f"truth is missing key: {key}")

        v = truth[key].detach().cpu().reshape(-1)

        vals.append(v)

        for local_id in range(v.numel()):
            names.append(f"{key}_{local_id}")

    return torch.cat(vals).numpy(), names


def get_truth_block_sum(model, truth):
    out = {}

    for item in model.decoder.param_specs:
        key = item["name"]

        if key not in truth:
            raise KeyError(f"truth is missing key: {key}")

        out[key] = to_numpy(truth[key])

    return out


# Use mcmc_ref if available; otherwise fall back to mcmc.
mcmc_obj = mcmc_ref if "mcmc_ref" in globals() else mcmc

vi_theta, vi_names, last_sum = get_vi_draws_and_summary(
    model=model,
    R=R_vi,
    beta_eps=beta_eps,
)

mcmc_theta = mcmc_obj["theta_draws"]
mcmc_names = mcmc_obj["theta_names"]

mcmc_sum = get_mcmc_block_summary(
    model=model,
    mcmc_obj=mcmc_obj,
    beta_eps=beta_eps,
)

theta_true, truth_names = get_truth_flat(model, truth)
truth_sum = get_truth_block_sum(model, truth)

assert vi_names == mcmc_names
assert vi_names == truth_names
assert vi_theta.shape[1] == mcmc_theta.shape[1]


# -----------------------------
# 3. Simulation / architecture / sparsity header
# -----------------------------

print("\n===== simulation and architecture summary =====")

n_train = X_train.shape[0] if "X_train" in globals() else None
n_val = X_val.shape[0] if "X_val" in globals() else None
n_test = X_test.shape[0] if "X_test" in globals() else None
p = X_train.shape[1] if "X_train" in globals() else model.decoder.input_dim

print("family:", getattr(model, "family", "unknown"))
print("sigma2:", sigma2 if "sigma2" in globals() else "not shown")
print("n_train:", n_train, "n_val:", n_val, "n_test:", n_test, "p:", p)

print("\nattention-FFN architecture:")
print(f"input embedding: {model.decoder.input_dim} -> {model.decoder.d_model}")
print("attention_type:", model.decoder.attention_type)
print("ffn_activation:", model.decoder.ffn_activation)

for spec in model.decoder.layers_spec:
    k = spec["block"]
    print(
        f"block {k}: "
        f"d_model={spec['d_model']} -> dff={spec['dff']} -> d_model={spec['d_model']} "
        f"with z_next = z + FFN(Attn(z))"
    )

print(f"output head: {model.decoder.d_model} -> {model.decoder.out_dim}")

print("\ndecoder dimensions:")
print("decoder dim:", model.decoder.dim)
print("s_dim:", model.decoder.s_dim)
print("u_dim:", model.decoder.u_dim)
print("t_dim:", model.decoder.t_dim)

truth_mask = np.abs(theta_true) > 1e-12
zero_mask = ~truth_mask

print("\ntruth sparsity:")
print("n_param:", int(theta_true.size))
print("n_nonzero:", int(truth_mask.sum()))
print("n_zero:", int(zero_mask.sum()))
print("truth density:", float(truth_mask.mean()))

if "sim_info" in globals():
    print("\nsimulation truth support:")
    if "active_idx" in sim_info:
        print("active input features:", np.asarray(sim_info["active_idx"]).tolist())
    if "active_state" in sim_info:
        print("active state coordinates:", np.asarray(sim_info["active_state"]).tolist())
    if "active_ff" in sim_info:
        print("active FFN units by block:")
        for k, aff in enumerate(sim_info["active_ff"]):
            print(f"  block {k}: {np.asarray(aff).tolist()}")

elif "E" in truth_sum:
    E_mask = np.abs(truth_sum["E"]) > 1e-12
    active_input = np.where(E_mask.any(axis=0))[0].tolist()
    active_state = np.where(E_mask.any(axis=1))[0].tolist()

    print("\ninput-side truth support from E:")
    print("active input features:", active_input)
    print("active state coordinates:", active_state)

print("\ntruth nonzero count by parameter block:")
truth_count_rows = []

for meta in get_param_meta(model.decoder):
    key = meta["param"]
    mask = np.abs(truth_sum[key]) > 1e-12

    truth_count_rows.append({
        "group": meta["group"],
        "param": key,
        "shape": str(meta["shape"]),
        "n_param": int(mask.size),
        "n_nonzero": int(mask.sum()),
        "density": float(mask.mean()),
    })

display(pd.DataFrame(truth_count_rows))


# -----------------------------
# 4. Posterior computation metrics: global
# -----------------------------

vi_pip = (np.abs(vi_theta) > beta_eps).mean(axis=0)
mcmc_pip = (np.abs(mcmc_theta) > beta_eps).mean(axis=0)

vi_mean = vi_theta.mean(axis=0)
mcmc_mean = mcmc_theta.mean(axis=0)

skl = np.full(vi_theta.shape[1], np.nan)

for j in np.where(truth_mask)[0]:
    skl[j] = hist_skl(vi_theta[:, j], mcmc_theta[:, j])

js = bern_js(vi_pip, mcmc_pip)

print("\n===== global VI vs MCMC posterior comparison =====")
print("beta_eps:", beta_eps)
print("R_vi:", R_vi)
print("R_mcmc:", mcmc_theta.shape[0])

global_metrics = pd.DataFrame([{
    "active_SKL_mean": float(np.nanmean(skl[truth_mask])),
    "active_SKL_median": float(np.nanmedian(skl[truth_mask])),
    "active_SKL_max": float(np.nanmax(skl[truth_mask])),
    "zero_JS_mean": float(np.mean(js[zero_mask])),
    "zero_JS_median": float(np.median(js[zero_mask])),
    "zero_JS_max": float(np.max(js[zero_mask])),
    "PIP_MAE_all": float(np.mean(np.abs(vi_pip - mcmc_pip))),
    "PIP_MAE_active": float(np.mean(np.abs(vi_pip[truth_mask] - mcmc_pip[truth_mask]))),
    "PIP_MAE_zero": float(np.mean(np.abs(vi_pip[zero_mask] - mcmc_pip[zero_mask]))),
    "mean_MAE_active": float(np.mean(np.abs(vi_mean[truth_mask] - mcmc_mean[truth_mask]))),
    "mean_MAE_zero": float(np.mean(np.abs(vi_mean[zero_mask] - mcmc_mean[zero_mask]))),
}])

display(global_metrics)


# -----------------------------
# 5. Posterior computation display by parameter block
#    Each cell: truth / MCMC posterior mean / LaST posterior mean
# -----------------------------

def triplet_value_df(key, nd=2):
    truth_val = to_numpy(truth_sum[key])
    mcmc_val = to_numpy(mcmc_sum[f"{key}_mean"])
    last_val = to_numpy(last_sum[f"{key}_mean"])

    if truth_val.ndim == 1:
        truth_val = truth_val[None, :]
        mcmc_val = mcmc_val[None, :]
        last_val = last_val[None, :]

    out = np.empty(truth_val.shape, dtype=object)

    for idx in np.ndindex(truth_val.shape):
        out[idx] = (
            f"{truth_val[idx]:.{nd}f}/"
            f"{mcmc_val[idx]:.{nd}f}/"
            f"{last_val[idx]:.{nd}f}"
        )

    row_prefix, col_prefix = param_label_prefix(key)
    return make_df(out, row_prefix=row_prefix, col_prefix=col_prefix)


def metric_row_for_param(key):
    meta = None

    for m in get_param_meta(model.decoder):
        if m["param"] == key:
            meta = m
            break

    start = meta["start"]
    end = meta["end"]

    idx = np.arange(start, end)

    local_truth = truth_mask[idx]
    local_zero = ~local_truth

    out = {
        "group": meta["group"],
        "param": key,
        "n_param": int(len(idx)),
        "n_active": int(local_truth.sum()),
        "n_zero": int(local_zero.sum()),
        "PIP_MAE": float(np.mean(np.abs(vi_pip[idx] - mcmc_pip[idx]))),
        "mean_MAE": float(np.mean(np.abs(vi_mean[idx] - mcmc_mean[idx]))),
    }

    if local_truth.any():
        out["active_SKL_mean"] = float(np.nanmean(skl[idx][local_truth]))
        out["active_SKL_median"] = float(np.nanmedian(skl[idx][local_truth]))
        out["active_PIP_MAE"] = float(
            np.mean(np.abs(vi_pip[idx][local_truth] - mcmc_pip[idx][local_truth]))
        )
    else:
        out["active_SKL_mean"] = np.nan
        out["active_SKL_median"] = np.nan
        out["active_PIP_MAE"] = np.nan

    if local_zero.any():
        out["zero_JS_mean"] = float(np.mean(js[idx][local_zero]))
        out["zero_PIP_MAE"] = float(
            np.mean(np.abs(vi_pip[idx][local_zero] - mcmc_pip[idx][local_zero]))
        )
    else:
        out["zero_JS_mean"] = np.nan
        out["zero_PIP_MAE"] = np.nan

    return out


print("\n===== parameter-wise posterior display =====")
print("cell format: truth / MCMC posterior mean / LaST posterior mean")

all_metric_rows = []

for group in ["input"] + [f"block_{k}" for k in range(model.decoder.n_blocks)] + ["output"]:
    group_metas = [m for m in get_param_meta(model.decoder) if m["group"] == group]

    if len(group_metas) == 0:
        continue

    print(f"\n========== {group} ==========")

    group_rows = []

    for meta in group_metas:
        key = meta["param"]

        print(f"\n--- {key}: truth / MCMC mean / LaST mean ---")
        display(triplet_value_df(key, nd=2))

        row = metric_row_for_param(key)
        group_rows.append(row)
        all_metric_rows.append(row)

    print("\nposterior-comparison metrics:")
    display(pd.DataFrame(group_rows))


print("\n===== all parameter-block posterior-comparison metrics =====")
metric_tbl = pd.DataFrame(all_metric_rows)
display(metric_tbl)


# -----------------------------
# 6. Selection summary by parameter block
#    No matrices; only counts.
# -----------------------------

def selection_count_row(key, pip_cut=0.5):
    truth_sel = np.abs(to_numpy(truth_sum[key])) > 1e-12
    mcmc_sel = to_numpy(mcmc_sum[f"{key}_epip"]) > pip_cut
    last_sel = to_numpy(last_sum[f"{key}_epip"]) > pip_cut

    truth_flat = truth_sel.reshape(-1)
    mcmc_flat = mcmc_sel.reshape(-1)
    last_flat = last_sel.reshape(-1)

    mcmc_tp = int((mcmc_flat & truth_flat).sum())
    mcmc_fp = int((mcmc_flat & ~truth_flat).sum())
    mcmc_fn = int((~mcmc_flat & truth_flat).sum())

    last_tp = int((last_flat & truth_flat).sum())
    last_fp = int((last_flat & ~truth_flat).sum())
    last_fn = int((~last_flat & truth_flat).sum())

    return {
        "param": key,
        "n_param": int(truth_flat.size),
        "truth_nonzero": int(truth_flat.sum()),
        "MCMC_selected": int(mcmc_flat.sum()),
        "LaST_selected": int(last_flat.sum()),
        "MCMC_TP": mcmc_tp,
        "MCMC_FP": mcmc_fp,
        "MCMC_FN": mcmc_fn,
        "LaST_TP": last_tp,
        "LaST_FP": last_fp,
        "LaST_FN": last_fn,
        "MCMC_only": int((mcmc_flat & ~last_flat).sum()),
        "LaST_only": int((last_flat & ~mcmc_flat).sum()),
        "both_selected": int((mcmc_flat & last_flat).sum()),
    }


print("\n===== element-level selection count summary =====")
print("selection rule: ePIP >", pip_cut)
print("counts are by parameter block; hidden/state labels are not treated as fixed identifiers")

selection_rows = []

for group in ["input"] + [f"block_{k}" for k in range(model.decoder.n_blocks)] + ["output"]:
    group_metas = [m for m in get_param_meta(model.decoder) if m["group"] == group]

    if len(group_metas) == 0:
        continue

    print(f"\n========== {group} ==========")

    rows = []

    for meta in group_metas:
        key = meta["param"]
        row = selection_count_row(key, pip_cut=pip_cut)

        rows.append(row)
        selection_rows.append({
            "group": group,
            **row,
        })

    display(pd.DataFrame(rows))


print("\n===== overall selection count summary =====")

sel_tbl = pd.DataFrame(selection_rows)

overall = pd.DataFrame([{
    "truth_nonzero": int(sel_tbl["truth_nonzero"].sum()),
    "MCMC_selected": int(sel_tbl["MCMC_selected"].sum()),
    "LaST_selected": int(sel_tbl["LaST_selected"].sum()),
    "MCMC_TP": int(sel_tbl["MCMC_TP"].sum()),
    "MCMC_FP": int(sel_tbl["MCMC_FP"].sum()),
    "MCMC_FN": int(sel_tbl["MCMC_FN"].sum()),
    "LaST_TP": int(sel_tbl["LaST_TP"].sum()),
    "LaST_FP": int(sel_tbl["LaST_FP"].sum()),
    "LaST_FN": int(sel_tbl["LaST_FN"].sum()),
    "MCMC_only": int(sel_tbl["MCMC_only"].sum()),
    "LaST_only": int(sel_tbl["LaST_only"].sum()),
    "both_selected": int(sel_tbl["both_selected"].sum()),
}])

overall["MCMC_precision"] = overall["MCMC_TP"] / np.maximum(
    overall["MCMC_TP"] + overall["MCMC_FP"], 1
)
overall["MCMC_recall"] = overall["MCMC_TP"] / np.maximum(
    overall["MCMC_TP"] + overall["MCMC_FN"], 1
)
overall["LaST_precision"] = overall["LaST_TP"] / np.maximum(
    overall["LaST_TP"] + overall["LaST_FP"], 1
)
overall["LaST_recall"] = overall["LaST_TP"] / np.maximum(
    overall["LaST_TP"] + overall["LaST_FN"], 1
)

display(overall)


===== simulation and architecture summary =====
family: gaussian
sigma2: 0.25
n_train: 96 n_val: 32 n_test: 32 p: 100

attention-FFN architecture:
input embedding: 100 -> 8
attention_type: self
ffn_activation: relu
block 0: d_model=8 -> dff=8 -> d_model=8 with z_next = z + FFN(Attn(z))
block 1: d_model=8 -> dff=8 -> d_model=8 with z_next = z + FFN(Attn(z))
output head: 8 -> 1

decoder dimensions:
decoder dim: 2222
s_dim: 1105
u_dim: 1105
t_dim: 12

truth sparsity:
n_param: 1105
n_nonzero: 105
n_zero: 1000
truth density: 0.09502262443438914

simulation truth support:
active input features: [1, 5, 58, 66]
active state coordinates: [0, 1, 2, 7]
active FFN units by block:
  block 0: [1, 2, 5, 6]
  block 1: [1, 2, 6, 7]

truth nonzero count by parameter block:


,group,param,shape,n_param,n_nonzero,density
0,input,E,"(8, 100)",800,16,0.02
1,input,e,"(8,)",8,4,0.50
2,block_0,W1_0,"(8, 8)",64,16,0.25
3,block_0,b1_0,"(8,)",8,4,0.50
4,block_0,W2_0,"(8, 8)",64,16,0.25
5,block_0,b2_0,"(8,)",8,4,0.50
6,block_1,W1_1,"(8, 8)",64,16,0.25
7,block_1,b1_1,"(8,)",8,4,0.50
8,block_1,W2_1,"(8, 8)",64,16,0.25
9,block_1,b2_1,"(8,)",8,4,0.50



===== global VI vs MCMC posterior comparison =====
beta_eps: 0.05
R_vi: 1000
R_mcmc: 8000


,active_SKL_mean,active_SKL_median,active_SKL_max,zero_JS_mean,zero_JS_median,zero_JS_max,PIP_MAE_all,PIP_MAE_active,PIP_MAE_zero,mean_MAE_active,mean_MAE_zero
0,0.417,0.113,9.187,0.083,0.099,0.276,0.267,0.112,0.283,0.063,0.041



===== parameter-wise posterior display =====
cell format: truth / MCMC posterior mean / LaST posterior mean

========== input ==========

--- E: truth / MCMC mean / LaST mean ---


...

,param,n_param,truth_nonzero,MCMC_selected,LaST_selected,MCMC_TP,MCMC_FP,MCMC_FN,LaST_TP,LaST_FP,LaST_FN,MCMC_only,LaST_only,both_selected
0,W1_1,64,16,0,0,0,0,16,0,0,16,0,0,0
1,b1_1,8,4,0,5,0,0,4,2,3,2,0,5,0
2,W2_1,64,16,0,0,0,0,16,0,0,16,0,0,0
3,b2_1,8,4,0,0,0,0,4,0,0,4,0,0,0



========== output ==========


,param,n_param,truth_nonzero,MCMC_selected,LaST_selected,MCMC_TP,MCMC_FP,MCMC_FN,LaST_TP,LaST_FP,LaST_FN,MCMC_only,LaST_only,both_selected
0,Wout,8,4,2,0,1,1,3,0,0,4,2,0,0
1,bout,1,1,0,0,0,0,1,0,0,1,0,0,0



===== overall selection count summary =====


,truth_nonzero,MCMC_selected,LaST_selected,MCMC_TP,MCMC_FP,MCMC_FN,LaST_TP,LaST_FP,LaST_FN,MCMC_only,LaST_only,both_selected,MCMC_precision,MCMC_recall,LaST_precision,LaST_recall
0,105,6,8,3,3,102,4,4,101,6,8,0,0.5,0.029,0.5,0.038
